In [9]:
import pandas as pd
import sqlite3

def extract():
    print("Starting ETL process ...")

    df = pd.read_csv('sales_data.csv')
    return df

In [10]:
def transform(df):
    print("Transforming the data ...")

    #converting order_time to datetime
    df['order_date'] = pd.to_datetime(df['order_date'])

    #For missing customer value enter 'Unknown'
    df['customer_name'] = df['customer_name'].fillna('Unknown')

    #cleaning country names
    df['country'] = df['country'].str.strip().str.title()

    #Converting quantity and unit_price to proper types
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0).astype(int)
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

    #calculate total amount
    df['total_amount'] = df['quantity'] * df['unit_price']

    #add year and month columns for analytics
    df['order_year'] = df['order_date'].dt.year
    df['order_month'] = df['order_date'].dt.strftime('%Y-%m')

    #filtering out invalid rows
    df = df[df['unit_price'] > 0]
    df = df[df['quantity'] > 0]

    return df 

In [11]:
def load(df):
    db_file = 'sales_database.db'

    #connecting to database
    conn = sqlite3.connect(db_file)

    #loading transformed data to database
    df.to_sql('sales', conn, if_exists='replace', index=False)
    
    conn.close()
    print(f"Data successfully loaded to {db_file}.")


In [12]:
def etl():
    extracted = extract()
    transformed = transform(extracted)
    load(transformed)

etl()

Starting ETL process ...
Transforming the data ...
Data successfully loaded to sales_database.db.
